In [1]:
import os
import glob
import cv2
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Folders to store single-nucleus cropped patches (32x32)
PATCH_SAVE_DIR = "dataset/classified_cells"
os.makedirs(f"{PATCH_SAVE_DIR}/train", exist_ok=True)
os.makedirs(f"{PATCH_SAVE_DIR}/test", exist_ok=True)

Using device: cpu


In [2]:
def extract_cell_crops(split_name="Train", crop_size=32):
    img_dir = f"dataset/CoNSeP/{split_name}/Images"
    mat_dir = f"dataset/CoNSeP/{split_name}/Labels"
    
    if not os.path.exists(img_dir):
        img_dir = f"dataset/CoNSeP/{split_name.lower()}/Images"
        mat_dir = f"dataset/CoNSeP/{split_name.lower()}/Labels"

    raw_images = sorted(glob.glob(os.path.join(img_dir, "*.png")))
    half = crop_size // 2
    cell_data = []

    for img_path in raw_images:
        base_id = os.path.splitext(os.path.basename(img_path))[0]
        mat_path = os.path.join(mat_dir, f"{base_id}.mat")
        
        if not os.path.exists(mat_path):
            continue
            
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        mat = sio.loadmat(mat_path)
        centroids = mat["inst_centroid"]
        types = mat["inst_type"].flatten()
        
        for center, label in zip(centroids, types):
            cy, cx = int(round(center[0])), int(round(center[1]))
            
            y1, y2 = max(0, cy - half), min(h, cy + half)
            x1, x2 = max(0, cx - half), min(w, cx + half)
            
            patch = img[y1:y2, x1:x2]
            if patch.shape[0] != crop_size or patch.shape[1] != crop_size:
                patch = cv2.resize(patch, (crop_size, crop_size))
                
            # Convert 1-indexed to 0-indexed and clamp safely between 0 and 4
            adjusted_label = int(np.clip(int(label) - 1, 0, 4))
            cell_data.append((patch, adjusted_label))

    print(f"Extracted {len(cell_data)} labeled nuclei crops for {split_name} set.")
    return cell_data

train_cells = extract_cell_crops("Train")
test_cells = extract_cell_crops("Test")

Extracted 15555 labeled nuclei crops for Train set.
Extracted 8777 labeled nuclei crops for Test set.


In [3]:
class NucleiCropDataset(Dataset):
    def __init__(self, cell_data, transform=None):
        self.cell_data = cell_data
        self.transform = transform

    def __len__(self):
        return len(self.cell_data)

    def __getitem__(self, idx):
        patch, label = self.cell_data[idx]
        
        # Normalize to [0, 1] float tensor
        patch = patch.astype(np.float32) / 255.0
        patch = np.transpose(patch, (2, 0, 1)) # (C, H, W)
        tensor_img = torch.tensor(patch, dtype=torch.float32)

        return tensor_img, torch.tensor(label, dtype=torch.long)

train_cls_dataset = NucleiCropDataset(train_cells)
test_cls_dataset  = NucleiCropDataset(test_cells)

train_cls_loader = DataLoader(train_cls_dataset, batch_size=32, shuffle=True)
test_cls_loader  = DataLoader(test_cls_dataset, batch_size=32, shuffle=False)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# 1. Define the Classifier Architecture
class NucleiClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# 2. Define Focal Loss for Class Imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# 3. Compute Class-Balanced Weights
all_train_labels = [label for _, label in train_cells]
class_counts = np.bincount(all_train_labels, minlength=5)
weights = 1.0 / (class_counts + 1e-5)
weights = torch.tensor(weights / weights.sum(), dtype=torch.float32).to(device)

# 4. Instantiate Model, Loss, and Optimizer
classifier = NucleiClassifier(num_classes=5).to(device)
criterion = FocalLoss(alpha=weights, gamma=2.0)
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)

print("Classifier initialized with Focal Loss and class balancing!")

Classifier initialized with Focal Loss and class balancing!


In [5]:
cls_epochs = 8
best_test_acc = 0.0

print("Starting Classifier Training...\n" + "=" * 50)

for epoch in range(1, cls_epochs + 1):
    classifier.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_cls_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        preds = classifier(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, predicted = preds.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100.0 * correct / total
    train_loss = running_loss / total

    # Validation
    classifier.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_cls_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = classifier(imgs)
            _, predicted = preds.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * val_correct / val_total
    print(f"Epoch [{epoch}/{cls_epochs}] | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(classifier.state_dict(), "models/classifier.pth")
        print(f"  --> Saved new best classifier to 'models/classifier.pth'")

print(f"\nTraining complete! Best Test Accuracy: {best_test_acc:.2f}%")

Starting Classifier Training...
Epoch [1/8] | Loss: 0.0040 | Train Acc: 4.82% | Test Acc: 6.76%
  --> Saved new best classifier to 'models/classifier.pth'
Epoch [2/8] | Loss: 0.0035 | Train Acc: 5.53% | Test Acc: 6.84%
  --> Saved new best classifier to 'models/classifier.pth'
Epoch [3/8] | Loss: 0.0034 | Train Acc: 6.17% | Test Acc: 6.39%
Epoch [4/8] | Loss: 0.0032 | Train Acc: 7.91% | Test Acc: 6.39%
Epoch [5/8] | Loss: 0.0030 | Train Acc: 8.16% | Test Acc: 7.63%
  --> Saved new best classifier to 'models/classifier.pth'
Epoch [6/8] | Loss: 0.0030 | Train Acc: 8.93% | Test Acc: 8.25%
  --> Saved new best classifier to 'models/classifier.pth'
Epoch [7/8] | Loss: 0.0028 | Train Acc: 9.86% | Test Acc: 7.26%
Epoch [8/8] | Loss: 0.0029 | Train Acc: 10.53% | Test Acc: 8.42%
  --> Saved new best classifier to 'models/classifier.pth'

Training complete! Best Test Accuracy: 8.42%
